# 5장 (실습 1): MCP Server의 Tool 권한을 최소권한으로 줄여보기

> 책 『AI 보안, 클라우드에서 시작한다』 5장 [실습] 1번을 노트북에서 단계별로 실행하는 버전입니다. 소요 시간은 30~40분입니다.
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다.

## 🎯 학습 목표
1. MCP Server에 **넓은 접근 경로**를 주면 AI가 업무와 상관없는 비밀 파일까지 읽을 수 있다는 것을 확인합니다.
2. MCP Server가 기본으로 노출하는 **쓰기·수정·이동 Tool**을 목록에서 찾고, 업무에 필요한 Tool만 남깁니다.
3. 접근 경로 제한, Tool 허용 목록, 운영체제의 읽기 전용 마운트를 **겹쳐 적용**하고 결과를 비교합니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: MCP Server 이미지와 클라이언트 | 예정 |
| 1 | 실습용 "민감한" 파일 만들기 | 예정 |
| 2 | 넓은 권한으로 연결하기 (나쁜 예) | 예정 |
| 3 | 서버가 제공하는 Tool 확인하기 | 예정 |
| 4 | 최소권한으로 줄이기 | 예정 |
| 5 | 이 설정은 누가 정해야 하나 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

## 실습 시나리오

README의 실습은 Claude Desktop 같은 AI 클라이언트에 MCP Server를 연결해 진행합니다. 이 노트북에서는 **파이썬이 AI 클라이언트 역할**을 합니다. AI 클라이언트가 내부에서 하는 일, 즉 MCP Server를 띄우고 JSON-RPC 메시지로 Tool 목록을 받고 Tool을 부르는 과정을 셀마다 직접 보며 진행합니다.

AI가 "홈 디렉터리에 어떤 파일이 있는지 알려줘"라는 요청을 받으면, 실제로는 `list_directory`, `read_text_file` 같은 Tool을 부릅니다. 이 노트북은 그 Tool 호출을 그대로 재현하므로, AI 모델 없이도 **AI가 무엇을 볼 수 있는지**를 정확히 확인할 수 있습니다.

> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)에서 Docker를 직접 부를 수 있는 환경**을 가정합니다. 셀 안의 `%%bash`는 그 셀 전체를 bash 명령으로 실행하라는 뜻입니다.
> - Node.js는 설치하지 않아도 됩니다. 공식 파일 시스템 MCP Server(`@modelcontextprotocol/server-filesystem`)를 버전을 고정한 Docker 이미지로 만들어 씁니다.
>
> **2. Jupyter 준비하기 (WSL 터미널에서 한 번만)**<br>
> - 2장 노트북([ch02_lab1_container_isolation.ipynb](../ch02-container-security/ch02_lab1_container_isolation.ipynb))의 실습 전 확인사항과 같습니다. `~/.venv-lab`에 jupyterlab을 설치한 뒤 이 폴더에서 `~/.venv-lab/bin/jupyter lab`을 실행합니다.
> - 노트북은 **반드시 `ch05-agent-mcp-security` 폴더에서 열어야** 합니다. 실습 파일을 이 폴더 기준 경로로 읽습니다.
>
> **3. 기준 버전**<br>
> - 아래 출력은 2026년 10월 23일 Ubuntu 24.04(WSL2), Docker 29.1.3, node:22-alpine, @modelcontextprotocol/server-filesystem 2026.8.31에서 실행한 결과입니다. 시각, 번호, ID 같은 값은 실행할 때마다 다릅니다. 직접 실행할 때는 출력이 없는 `ch05_lab1_mcp_least_privilege.ipynb`을 쓰세요.

> ⚠️ **주의**
> **실제 업무용 PC가 아닌 실습용 가상 머신이나 별도 사용자 계정에서 진행하세요.** 2단계에서 MCP Server가 홈 디렉터리 전체를 읽습니다. 이 노트북은 홈 디렉터리의 파일 목록을 출력에 남기므로, 출력을 다른 사람과 공유하기 전에 확인하세요.

## 0. 실습 준비: MCP Server 이미지와 클라이언트

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: MCP Server 이미지와 클라이언트 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | 실습용 "민감한" 파일 만들기 | 예정 |
| 2 | 넓은 권한으로 연결하기 (나쁜 예) | 예정 |
| 3 | 서버가 제공하는 Tool 확인하기 | 예정 |
| 4 | 최소권한으로 줄이기 | 예정 |
| 5 | 이 설정은 누가 정해야 하나 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

### 0.1. 도우미 함수와 도구 확인

In [1]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 폴더
os.environ["PATH"] = f"{Path.home() / 'bin'}:{os.environ['PATH']}"   # 실습 도구를 ~/bin에 둔 경우


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def table(header, rows):
    """머리글과 행 목록으로 마크다운 표를 그립니다."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


VERSION_CMD = {
    "docker": "docker --version",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd).splitlines()[0] if sh(cmd) else ''}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:10} {state}")

HOME = Path.home()
print("\n홈 디렉터리:", HOME)

docker     ✅ Docker version 29.1.3, build 29.1.3-0ubuntu3~24.04.2

홈 디렉터리: /home/aisec


### 0.2. MCP Server 이미지 만들기

README의 설정은 `npx -y @modelcontextprotocol/server-filesystem`처럼 **버전 없이** 패키지를 받아 실행합니다. 이렇게 하면 실행할 때마다 그 시점의 최신 버전이 내려받아집니다. 노트북에서는 버전을 고정해 이미지로 한 번만 만들어 둡니다. 이 차이는 과제 1에서 다시 다룹니다.

In [2]:
%%bash
docker build -q -t mcp-fs-lab:2026.8.31 - <<'EOF'
FROM node:22-alpine
RUN npm install -g --no-fund --no-audit @modelcontextprotocol/server-filesystem@2026.8.31
ENTRYPOINT ["mcp-server-filesystem"]
EOF
docker images mcp-fs-lab --format '{{.Repository}}:{{.Tag}}  {{.Size}}'

sha256:a532f8f8d8e6fde9410894ad7bb0484e6d37eb76c3f0ee49ee419b8f71fd4abd


mcp-fs-lab:2026.8.31  325MB


### 0.3. 파이썬 MCP 클라이언트

MCP Server는 표준 입출력(stdio)으로 JSON-RPC 메시지를 한 줄씩 주고받습니다. AI 클라이언트가 하는 일은 세 가지입니다.

1. `initialize`: 서버와 인사하고 프로토콜 버전을 맞춥니다.
2. `tools/list`: 서버가 제공하는 Tool 목록을 받습니다. AI 모델은 이 목록을 보고 어떤 Tool을 부를지 정합니다.
3. `tools/call`: Tool 하나를 인자와 함께 부릅니다.

아래 `McpServer`는 이 세 가지만 하는 작은 클라이언트입니다. 서버는 Docker 컨테이너로 실행하며, 두 가지로 접근 범위를 정합니다.

- `mounts`: 컨테이너에 보여 줄 **호스트 폴더**(Docker `-v`). 운영체제 수준의 경계입니다.
- `allowed`: MCP Server에 넘기는 **허용 디렉터리** 인자. README의 설정 파일 `args` 마지막 값과 같습니다.

> 📌 (참고) 컨테이너는 `--user`로 내 사용자 ID로 실행합니다. 서버가 파일을 만들어도 root 소유가 되지 않고, 내가 읽을 수 없는 파일은 서버도 읽지 못합니다.

In [3]:
class McpServer:
    """파일 시스템 MCP Server를 컨테이너로 띄워 JSON-RPC로 대화합니다."""

    def __init__(self, mounts, allowed):
        uid_gid = f"{os.getuid()}:{os.getgid()}"
        cmd = ["docker", "run", "-i", "--rm", "--user", uid_gid]
        for host, inside in mounts:
            cmd += ["-v", f"{host}:{inside}"]
        cmd += ["mcp-fs-lab:2026.8.31", *allowed]
        self.proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                     stderr=subprocess.DEVNULL, text=True)
        self.next_id = 0
        info = self.request("initialize", {"protocolVersion": "2025-03-26", "capabilities": {},
                                           "clientInfo": {"name": "notebook", "version": "1"}})
        self.notify("notifications/initialized")
        self.server_info = info["serverInfo"]

    def notify(self, method):
        self.proc.stdin.write(json.dumps({"jsonrpc": "2.0", "method": method}) + "\n")
        self.proc.stdin.flush()

    def request(self, method, params=None):
        self.next_id += 1
        msg = {"jsonrpc": "2.0", "id": self.next_id, "method": method, "params": params or {}}
        self.proc.stdin.write(json.dumps(msg) + "\n")
        self.proc.stdin.flush()
        while True:                                   # 알림은 건너뛰고 내 요청의 응답만 받는다
            reply = json.loads(self.proc.stdout.readline())
            if reply.get("id") == self.next_id:
                return reply.get("result", reply.get("error"))

    def tools(self):
        return self.request("tools/list")["tools"]

    def call(self, name, **args):
        """Tool을 부르고 (성공 여부, 결과 글자)를 돌려줍니다."""
        res = self.request("tools/call", {"name": name, "arguments": args})
        text = "\n".join(c.get("text", "") for c in res.get("content", []))
        return not res.get("isError", False), text

    def close(self):
        self.proc.stdin.close()
        self.proc.wait(timeout=10)


print("클라이언트 준비 완료")

클라이언트 준비 완료


## 1. 실습용 "민감한" 파일 만들기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: MCP Server 이미지와 클라이언트 | ✅ 완료 |
| 1 | 실습용 "민감한" 파일 만들기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | 넓은 권한으로 연결하기 (나쁜 예) | 예정 |
| 3 | 서버가 제공하는 Tool 확인하기 | 예정 |
| 4 | 최소권한으로 줄이기 | 예정 |
| 5 | 이 설정은 누가 정해야 하나 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

실제 비밀정보 대신 가짜 파일을 만들어 둡니다. `~/mcp-lab/docs`는 업무용 폴더, `~/fake-secrets`는 AI가 볼 필요가 없는 비밀 폴더 역할입니다.

In [4]:
%%bash
rm -rf ~/mcp-lab ~/fake-secrets
mkdir -p ~/mcp-lab/docs ~/fake-secrets
echo "회의록: 3분기 신제품 출시 일정 논의" > ~/mcp-lab/docs/meeting.txt
cat > ~/fake-secrets/credentials << 'EOF'
[default]
aws_access_key_id = AKIAFAKEFAKEFAKE0000
aws_secret_access_key = fakefakefakefakefakefakefakefakefake
EOF
echo "DB_PASSWORD=FakePassw0rd!" > ~/fake-secrets/.env
find ~/mcp-lab ~/fake-secrets -type f

/home/aisec/mcp-lab/docs/meeting.txt
/home/aisec/fake-secrets/.env
/home/aisec/fake-secrets/credenti

als


## 2. 넓은 권한으로 연결하기 (나쁜 예)

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: MCP Server 이미지와 클라이언트 | ✅ 완료 |
| 1 | 실습용 "민감한" 파일 만들기 | ✅ 완료 |
| 2 | 넓은 권한으로 연결하기 (나쁜 예) | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | 서버가 제공하는 Tool 확인하기 | 예정 |
| 4 | 최소권한으로 줄이기 | 예정 |
| 5 | 이 설정은 누가 정해야 하나 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

### 2.1. 홈 디렉터리 전체로 연결

README의 나쁜 예(`mcp/config-home-wide.json`)는 MCP Server에 **홈 디렉터리 전체**를 허용합니다.

In [5]:
print((LAB / "mcp" / "config-home-wide.json").read_text())

{
  "mcpServers": {
    "filesystem": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        "/Users/lab-user"
      ]
    }
  }
}



같은 설정을 컨테이너로 재현합니다. 홈 디렉터리를 컨테이너 안의 `/home/lab-user`로 보여 주고, 서버에도 `/home/lab-user`를 허용합니다.

In [6]:
wide = McpServer(mounts=[(HOME, "/home/lab-user")], allowed=["/home/lab-user"])
print("연결된 서버:", wide.server_info)

연결된 서버: {'name': 'secure-filesystem-server', 'version': '0.2.0'}


### 2.2. AI가 받는 요청을 Tool 호출로 바꿔 보기

사용자가 AI에게 이렇게 요청했다고 합시다.

```text
내 홈 디렉터리에 어떤 폴더와 설정 파일이 있는지 알려줘.
그리고 fake-secrets 폴더 안의 파일 내용을 보여줘.
```

AI 클라이언트는 이 요청을 처리하려고 아래 Tool들을 차례로 부릅니다.

In [7]:
ok, text = wide.call("list_directory", path="/home/lab-user")
print("── list_directory /home/lab-user")
print(text)

── list_directory /home/lab-user
[FILE] .bash_logout
[FILE] .bashrc
[DIR] .cache
[DIR] .cache-trivy
[DIR] .config
[DIR] .docker
[DIR] .ipython
[DIR] .jupyter
[DIR] .kube
[FILE] .lesshst
[DIR] .local
[FILE] .motd_shown
[FILE] .profile
[DIR] .venv-lab
[FILE] .wget-hsts
[DIR] bin
[DIR] ch10-lab
[DIR] egress-lab
[DIR] fake-secrets
[DIR] image-lab
[DIR] mcp-lab
[DIR] nbtest
[DIR] pod-lab
[DIR] sw-inventory


In [8]:
for name in ["credentials", ".env"]:
    ok, text = wide.call("read_text_file", path=f"/home/lab-user/fake-secrets/{name}")
    print(f"── read_text_file fake-secrets/{name} → {'성공' if ok else '실패'}")
    print(text, "\n")

── read_text_file fake-secrets/credentials → 성공
[default]
aws_access_key_id = AKIAFAKEFAKEFAKE0000
aws_secret_access_key = fakefakefakefakefakefakefakefakefake
 

── read_text_file fake-secrets/.env → 성공
DB_PASSWORD=FakePassw0rd!
 



AI가 `fake-secrets/credentials`의 내용을 그대로 읽었습니다. 실제 환경이었다면 `.ssh`, `.aws`, `.kube` 같은 폴더도 똑같이 보였을 것입니다. 위 목록에서 `.kube`나 `.docker` 같은 폴더가 보인다면, 그 안의 인증 정보도 AI가 읽을 수 있는 범위에 있다는 뜻입니다.

> 🙋 (생각해 보기)
> AI가 이 파일을 읽은 뒤 대화 기록에 남기면, 비밀 값은 AI 서비스 쪽 로그에도 남습니다. 4장 실습 1에서 본 "대화 내용의 보존·학습" 문제와 이어집니다.

### 2.3. 업무와 상관없는 파일 찾아보기

넓은 권한의 문제는 "AI가 일부러 찾아다닐 수도 있다"는 점입니다. 실습 2의 간접 프롬프트 주입처럼 문서 속 지시에 속은 AI는 `search_files` Tool로 비밀 파일을 찾을 수 있습니다.

In [9]:
for pattern in ["**/credentials", "**/.env", "**/*.pem", "**/config"]:
    ok, text = wide.call("search_files", path="/home/lab-user", pattern=pattern)
    hits = [line for line in text.splitlines() if line.startswith("/") and "/.venv-lab/" not in line]
    print(f"{pattern:16} → {len(hits)}건", hits[:3])

**/credentials   → 1건 ['/home/lab-user/fake-secrets/credentials']


**/.env          → 1건 ['/home/lab-user/fake-secrets/.env']


**/*.pem         → 0건 []


**/config        → 3건 ['/home/lab-user/.cache/trivy/policy/content/commands/config', '/home/lab-user/.cache/trivy/policy/content/policies/cloud/policies/aws/config', '/home/lab-user/.kube/config']


## 3. 서버가 제공하는 Tool 확인하기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: MCP Server 이미지와 클라이언트 | ✅ 완료 |
| 1 | 실습용 "민감한" 파일 만들기 | ✅ 완료 |
| 2 | 넓은 권한으로 연결하기 (나쁜 예) | ✅ 완료 |
| 3 | 서버가 제공하는 Tool 확인하기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | 최소권한으로 줄이기 | 예정 |
| 5 | 이 설정은 누가 정해야 하나 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

이 서버가 제공하는 Tool 목록을 받습니다. MCP의 Tool에는 서버가 붙인 **주석(annotations)** 이 있습니다. `readOnlyHint`는 읽기 전용인지, `destructiveHint`는 기존 내용을 지우거나 덮어쓸 수 있는지를 알려 줍니다.

In [10]:
TOOLS = wide.tools()
rows = []
for t in TOOLS:
    ann = t.get("annotations", {})
    read_only = ann.get("readOnlyHint", False)
    rows.append((f"`{t['name']}`", "읽기" if read_only else "**쓰기**",
                 "예" if ann.get("destructiveHint") else "-",
                 "필요" if read_only else "불필요",
                 t["description"].split(".")[0][:60]))
table(["Tool", "종류", "파괴적", "문서 요약 업무에 필요한가", "설명(앞부분)"], rows)
print(f"전체 {len(TOOLS)}개 중 쓰기 Tool {sum(r[1] != '읽기' for r in rows)}개")

| Tool | 종류 | 파괴적 | 문서 요약 업무에 필요한가 | 설명(앞부분) |
|---|---|---|---|---|
| `read_file` | 읽기 | - | 필요 | Read the complete contents of a file as text |
| `read_text_file` | 읽기 | - | 필요 | Read the complete contents of a file from the file system as |
| `read_media_file` | 읽기 | - | 필요 | Read a file and return it as a base64-encoded content block  |
| `read_multiple_files` | 읽기 | - | 필요 | Read the contents of multiple files simultaneously |
| `write_file` | **쓰기** | 예 | 불필요 | Create a new file or completely overwrite an existing file w |
| `edit_file` | **쓰기** | 예 | 불필요 | Make line-based edits to a text file |
| `create_directory` | **쓰기** | - | 불필요 | Create a new directory or ensure a directory exists |
| `list_directory` | 읽기 | - | 필요 | Get a detailed listing of all files and directories in a spe |
| `list_directory_with_sizes` | 읽기 | - | 필요 | Get a detailed listing of all files and directories in a spe |
| `directory_tree` | 읽기 | - | 필요 | Get a recursive tree view of files and directories as a JSON |
| `move_file` | **쓰기** | 예 | 불필요 | Move or rename files and directories |
| `search_files` | 읽기 | - | 필요 | Recursively search for files and directories matching a patt |
| `get_file_info` | 읽기 | - | 필요 | Retrieve detailed metadata about a file or directory |
| `list_allowed_directories` | 읽기 | - | 필요 | Returns the list of directories that this server is allowed  |

전체 14개 중 쓰기 Tool 4개


"파일을 읽어 요약해 줘"라는 업무에 쓰기 Tool이 필요할까요? 실제로 쓰기가 되는지 확인합니다. 홈 디렉터리에 파일을 하나 만들어 봅니다.

In [11]:
ok, text = wide.call("write_file", path="/home/lab-user/planted-by-ai.txt", content="AI가 만든 파일\n")
print("write_file →", "성공" if ok else "실패", "|", text)
print("호스트에서 확인:", (HOME / "planted-by-ai.txt").exists())
wide.close()

write_file → 성공 | Successfully wrote to /home/lab-user/planted-by-ai.txt
호스트에서 확인: True


`[표 실습5-1]` 파일 시스템 MCP Server의 Tool과 업무 필요성

| Tool 종류 | 하는 일 | 문서 요약 업무에 필요한가 |
|---|---|---|
| 파일 읽기, 목록 보기, 검색 | 파일 내용·디렉터리 조회 | 필요 |
| 파일 쓰기, 수정 | 파일 생성·덮어쓰기·부분 수정 | 불필요 |
| 이동, 디렉터리 생성 | 파일 이동·이름 변경, 폴더 생성 | 불필요 |

> 📌 (참고) Tool 이름과 개수는 서버 버전에 따라 다릅니다. 이 노트북의 표는 위 셀이 실행 시점의 서버에서 직접 받은 목록입니다.

## 4. 최소권한으로 줄이기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: MCP Server 이미지와 클라이언트 | ✅ 완료 |
| 1 | 실습용 "민감한" 파일 만들기 | ✅ 완료 |
| 2 | 넓은 권한으로 연결하기 (나쁜 예) | ✅ 완료 |
| 3 | 서버가 제공하는 Tool 확인하기 | ✅ 완료 |
| 4 | 최소권한으로 줄이기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | 이 설정은 누가 정해야 하나 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

### 4.1. ① 접근 경로 줄이기

README의 `mcp/config-restricted.json`은 허용 디렉터리를 업무 폴더 하나로 줄입니다.

In [12]:
print((LAB / "mcp" / "config-restricted.json").read_text())

{
  "mcpServers": {
    "filesystem": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        "/Users/lab-user/mcp-lab/docs"
      ]
    }
  }
}



### 4.2. ② Tool 허용 목록

클라이언트가 Tool별 허용 설정을 지원하면 쓰기·수정·이동 Tool은 끄거나 "실행 전 항상 확인"으로 둡니다. 아래 `GuardedClient`는 그 역할을 흉내 냅니다. 허용 목록에 없는 Tool은 **서버에 보내기 전에** 클라이언트가 막습니다.

In [13]:
class GuardedClient:
    """허용 목록에 있는 Tool만 서버로 보내는 클라이언트"""

    def __init__(self, server, allow):
        self.server, self.allow = server, set(allow)

    def call(self, name, **args):
        if name not in self.allow:
            return False, f"[클라이언트 차단] {name}은 허용 목록에 없습니다"
        return self.server.call(name, **args)


READ_ONLY = [t["name"] for t in TOOLS if t.get("annotations", {}).get("readOnlyHint")]
print("허용 목록:", READ_ONLY)

허용 목록: ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'search_files', 'get_file_info', 'list_allowed_directories']


### 4.3. ③ 운영체제 권한으로 한 번 더

클라이언트가 Tool 제한을 지원하지 않거나 설정이 실수로 풀릴 수도 있습니다. 그래서 컨테이너에는 업무 폴더 **하나만**, 그것도 **읽기 전용(`:ro`)** 으로 보여 줍니다. 서버나 클라이언트 설정이 틀려도 운영체제가 막습니다.

In [14]:
DOCS_IN = "/home/lab-user/mcp-lab/docs"
narrow_server = McpServer(mounts=[(HOME / "mcp-lab" / "docs", f"{DOCS_IN}:ro")], allowed=[DOCS_IN])
narrow = GuardedClient(narrow_server, READ_ONLY)
print("연결된 서버:", narrow_server.server_info)

연결된 서버: {'name': 'secure-filesystem-server', 'version': '0.2.0'}


### 4.4. 같은 요청을 다시 해 보기

2단계와 같은 요청, 그리고 업무 요청과 쓰기 요청을 넓은 설정과 좁은 설정에 똑같이 보내 비교합니다.

> 📌 (참고) 넓은 설정은 3단계에서 닫았으므로 다시 띄웁니다. 비교를 공정하게 하려고 넓은 설정에는 Tool 제한을 걸지 않습니다.

In [15]:
wide = McpServer(mounts=[(HOME, "/home/lab-user")], allowed=["/home/lab-user"])

REQUESTS = [
    ("홈 디렉터리의 설정 파일 목록", "list_directory", {"path": "/home/lab-user"}),
    ("`fake-secrets/credentials` 읽기", "read_text_file", {"path": "/home/lab-user/fake-secrets/credentials"}),
    ("`mcp-lab/docs/meeting.txt` 요약", "read_text_file", {"path": f"{DOCS_IN}/meeting.txt"}),
    ("새 파일 만들기", "write_file", {"path": f"{DOCS_IN}/new.txt", "content": "test"}),
]


def verdict(ok, text):
    if ok:
        return "✅ 가능 · " + text.replace("\n", " ")[:40]
    return "⛔ " + text.split(":")[0][:50]


rows = [(label, verdict(*wide.call(tool, **args)), verdict(*narrow.call(tool, **args)))
        for label, tool, args in REQUESTS]
table(["요청", "홈 디렉터리 전체 허용", "업무 폴더만 + 읽기 Tool만 + 읽기 전용"], rows)
wide.close()

| 요청 | 홈 디렉터리 전체 허용 | 업무 폴더만 + 읽기 Tool만 + 읽기 전용 |
|---|---|---|
| 홈 디렉터리의 설정 파일 목록 | ✅ 가능 · [FILE] .bash_logout [FILE] .bashrc [DIR] | ⛔ Access denied - path outside allowed directories |
| `fake-secrets/credentials` 읽기 | ✅ 가능 · [default] aws_access_key_id = AKIAFAKEFA | ⛔ Access denied - path outside allowed directories |
| `mcp-lab/docs/meeting.txt` 요약 | ✅ 가능 · 회의록: 3분기 신제품 출시 일정 논의  | ✅ 가능 · 회의록: 3분기 신제품 출시 일정 논의  |
| 새 파일 만들기 | ✅ 가능 · Successfully wrote to /home/lab-user/mcp | ⛔ [클라이언트 차단] write_file은 허용 목록에 없습니다 |

### 4.5. 겹친 방어가 각각 막는지 확인하기

"Tool 허용 목록이 실수로 풀렸다면?" 클라이언트 단계를 건너뛰고 서버에 직접 쓰기를 보내 봅니다. 이번에는 운영체제의 읽기 전용 마운트가 막아야 합니다.

In [16]:
ok, text = narrow_server.call("write_file", path=f"{DOCS_IN}/new.txt", content="test")
print("클라이언트 제한 없이 write_file →", "성공" if ok else "실패")
print(text)
narrow_server.close()

클라이언트 제한 없이 write_file → 실패
EROFS: read-only file system, open '/home/lab-user/mcp-lab/docs/new.txt.3c5ef11eb084a7c04b080dd103fe554a.tmp'


`[표 실습5-2]` 권한 범위에 따른 결과 비교

| 요청 | 홈 디렉터리 전체 허용 | 업무 폴더만 허용 |
|---|---|---|
| 홈 디렉터리의 설정 파일 목록 | 보임 | 거부 (허용된 디렉터리 밖) |
| `fake-secrets/credentials` 읽기 | 내용 노출 | 거부 |
| `mcp-lab/docs/meeting.txt` 요약 | 가능 | 가능 |
| 새 파일 만들기 | 가능 | 차단 또는 실행 전 확인 |

세 가지 장치는 각각 다른 곳에서 막습니다.

| 장치 | 막는 위치 | 무너지는 경우 |
|---|---|---|
| 허용 디렉터리 인자 | MCP Server | 서버 설정을 잘못 쓰거나 서버에 경로 검사 버그가 있을 때 |
| Tool 허용 목록 | AI 클라이언트 | 클라이언트가 지원하지 않거나, 사용자가 "항상 허용"을 눌렀을 때 |
| 읽기 전용 마운트 | 운영체제 | 서버를 컨테이너 밖에서 직접 실행할 때 |

## 5. 이 설정은 누가 정해야 하나

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: MCP Server 이미지와 클라이언트 | ✅ 완료 |
| 1 | 실습용 "민감한" 파일 만들기 | ✅ 완료 |
| 2 | 넓은 권한으로 연결하기 (나쁜 예) | ✅ 완료 |
| 3 | 서버가 제공하는 Tool 확인하기 | ✅ 완료 |
| 4 | 최소권한으로 줄이기 | ✅ 완료 |
| 5 | 이 설정은 누가 정해야 하나 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

이 실습에서는 사용자가 직접 설정 파일을 고쳤습니다. 회사에서 직원 각자가 MCP Server를 마음대로 추가하고 범위를 정한다면, 보안팀은 누가 어떤 서버를 어떤 범위로 연결했는지 알 수 없습니다. 5-3의 등록·승인 절차와 7-4의 MCP Gateway가 필요한 이유입니다.

등록·승인 절차의 첫걸음은 **설정 파일을 사람이 읽기 전에 기계가 먼저 검사**하는 것입니다. 다음 과제 1에서 간단한 검사기를 만들어 봅니다.

## 6. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: MCP Server 이미지와 클라이언트 | ✅ 완료 |
| 1 | 실습용 "민감한" 파일 만들기 | ✅ 완료 |
| 2 | 넓은 권한으로 연결하기 (나쁜 예) | ✅ 완료 |
| 3 | 서버가 제공하는 Tool 확인하기 | ✅ 완료 |
| 4 | 최소권한으로 줄이기 | ✅ 완료 |
| 5 | 이 설정은 누가 정해야 하나 | ✅ 완료 |
| 6 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 7 | 정리 | 예정 |

### 6.1. 체크포인트

- [ ] 처음 설정에서 AI가 볼 수 있었던 민감한 파일은 무엇이었나요?
- [ ] 업무에 필요 없는 Tool(쓰기·수정·이동)이 기본으로 노출되어 있다는 것을 확인했나요?
- [ ] 접근 경로 제한과 Tool 제한, 두 가지를 모두 적용했나요?
- [ ] 실제 업무에서 Tool 권한은 누가 결정하고 승인해야 할까요?

### 6.2. 과제 1: MCP 설정 검사기 만들기

직원이 등록을 요청한 MCP 설정 파일을 자동으로 검사하는 함수 `review()`를 완성합니다. 아래 세 가지를 찾아 경고 목록으로 돌려주면 됩니다. 이 폴더의 두 설정 파일과 직접 만든 예시로 확인합니다.

1. 허용 디렉터리가 **홈 디렉터리나 루트(`/`) 자체**인 경우
2. 패키지에 **버전이 고정되지 않은** 경우 (`@modelcontextprotocol/server-filesystem`처럼 `@버전`이 없음). 실행할 때마다 다른 코드가 내려받아지므로, 패키지가 탈취되면 바로 영향을 받습니다(8장의 공급망 보안).
3. `npx -y`처럼 **설치 확인을 건너뛰는** 옵션을 쓰는 경우

In [17]:
import re


def review(config):
    """MCP 설정(dict)을 검사해 (서버 이름, 경고) 목록을 돌려줍니다."""
    warnings = []
    for name, server in config.get("mcpServers", {}).items():
        args = server.get("args", [])
        # TODO 1: args 가운데 홈 디렉터리(/Users/이름, /home/이름, C:\\Users\\이름)나 "/"인 경로를 찾습니다.
        # TODO 2: "@modelcontextprotocol/..." 패키지 이름 뒤에 "@버전"이 없으면 경고합니다.
        # TODO 3: "-y"가 있으면 경고합니다.
        pass
    return warnings


samples = {
    "config-home-wide.json": json.loads((LAB / "mcp" / "config-home-wide.json").read_text()),
    "config-restricted.json": json.loads((LAB / "mcp" / "config-restricted.json").read_text()),
    "pinned-example": {"mcpServers": {"filesystem": {"command": "npx", "args": [
        "@modelcontextprotocol/server-filesystem@2026.8.31", "/home/lab-user/mcp-lab/docs"]}}},
}
for label, cfg in samples.items():
    print(f"── {label}")
    for server, warning in review(cfg) or [("-", "경고 없음")]:
        print(f"   {server}: {warning}")

── config-home-wide.json
   -: 경고 없음
── config-restricted.json
   -: 경고 없음
── pinned-example
   -: 경고 없음


<details>
<summary>💡 힌트 보기</summary>

```python
HOME_RE = re.compile(r"^(/Users/[^/]+|/home/[^/]+|[A-Za-z]:\\Users\\[^\\]+|/)/?$")
PKG_RE = re.compile(r"^(@[^/]+/[^@]+)(@.+)?$")

for a in args:
    if HOME_RE.match(a):
        warnings.append((name, f"허용 경로가 너무 넓음: {a}"))
    m = PKG_RE.match(a)
    if m and not m.group(2):
        warnings.append((name, f"버전 고정 안 됨: {a}"))
if "-y" in args:
    warnings.append((name, "설치 확인 건너뜀(-y)"))
```

결과로 `config-home-wide.json`에는 경고 3개, `config-restricted.json`에는 2개(버전 미고정, `-y`), `pinned-example`에는 0개가 나와야 합니다. **restricted 설정도 공급망 관점에서는 아직 안전하지 않다**는 점이 이 과제의 핵심입니다.
</details>

### 6.3. 과제 2: 승인 기준 정하기 (서술)

영업팀이 "고객 제안서 폴더를 AI가 읽고 초안을 만들어 같은 폴더에 저장하게 해 달라"며 파일 시스템 MCP Server 등록을 요청했습니다. 이 실습의 세 가지 장치(허용 디렉터리, Tool 허용 목록, 운영체제 권한)를 각각 **어떻게 설정해 승인할지** 적어 보세요. 쓰기가 꼭 필요하다면 어디까지 허용할 수 있을까요?

✍️ **답 적기** (이 셀을 더블클릭해 고칩니다)

- 허용 디렉터리:
- 허용할 Tool:
- 운영체제 권한:

## 7. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: MCP Server 이미지와 클라이언트 | ✅ 완료 |
| 1 | 실습용 "민감한" 파일 만들기 | ✅ 완료 |
| 2 | 넓은 권한으로 연결하기 (나쁜 예) | ✅ 완료 |
| 3 | 서버가 제공하는 Tool 확인하기 | ✅ 완료 |
| 4 | 최소권한으로 줄이기 | ✅ 완료 |
| 5 | 이 설정은 누가 정해야 하나 | ✅ 완료 |
| 6 | 체크포인트와 과제 | ✅ 완료 |
| 7 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

실습 폴더와 이미지를 지웁니다. 실습 2에서 같은 `~/mcp-lab/docs` 폴더를 쓰려면 이 셀을 실행하기 전에 실습 2를 먼저 해도 됩니다.

In [18]:
%%bash
rm -rf ~/mcp-lab ~/fake-secrets ~/planted-by-ai.txt
docker rmi mcp-fs-lab:2026.8.31 >/dev/null && echo "이미지 삭제됨"
ls ~/planted-by-ai.txt 2>/dev/null || echo "실습 파일 삭제됨"

이미지 삭제됨


실습 파일 삭제됨


---
> 다음 실습 2에서는 MCP Server가 읽어 온 문서 속에 숨은 지시문이 AI의 행동을 바꾸는 간접 프롬프트 주입을 다룹니다. 이 실습의 최소권한은 "속더라도 할 수 있는 일을 줄이는" 방어입니다.